<a href="https://colab.research.google.com/github/snehangshu1/-Building-UI-for-Study-Assistant-with-Gradio/blob/main/SkillMap_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
!pip install -qU langchain

In [16]:
!pip install -U langchain-google-genai

In [17]:
from langchain.agents import create_agent

In [18]:
from langchain.chat_models import init_chat_model
from google.colab import userdata
from pprint import pprint

In [19]:
google_api_key = userdata.get('api_key')
model = init_chat_model(
    "google_genai:gemini-2.5-flash",
    api_key=google_api_key
)

In [20]:
!pip install -qU langchain-tavily

In [21]:
from langchain_tavily import TavilySearch

tavily_api_key=userdata.get('tavily_api')

skill_demand_tool = TavilySearch(
    max_results=5,
    search_depth="advanced",
    tavily_api_key=tavily_api_key
)


In [22]:
from langchain.tools import tool

In [23]:
@tool
def search_jobs(skill: str, location: str)-> list:
  """Search fro jobs based on provided skill and location"""
  print(f"Job search has been started")
  print(f"Searching job for {skill} in {location}")

  import requests

  url = "https://jsearch.p.rapidapi.com/search"

  querystring = {
        "query": f"{skill} in {location}",
        "page": "1",
        "country": "in",
        "employment_types": "INTERN,FULLTIME",
        "job_requirements": "no_experience,under_3_years_experience"
    }

  headers = {
	"x-rapidapi-key": "1237f2b317msh7479c7de33807c9p18bf81jsn42398dfe8a9b",
	"x-rapidapi-host": "jsearch.p.rapidapi.com",
	"Content-Type": "application/json"
}

  response = requests.get(url, headers=headers, params=querystring)
  data=response.json()
  jobs=data.get("data",[])

  current_job_openings=[]
  for job in jobs:
    current_job_openings.append({
        "company_name":job.get("employer_name"),
        "job_title":job.get("job_title"),
        "job_description":job.get("job_description"),
        "location":job.get("job_location"),
        "direct_apply_link":job.get("direct_apply_link"),
        "employment_type":job.get("job_employemnt_type"),
        "job_posted_at":job.get("job_posted_at")

    })
  print(f"{len(current_job_openings)} Jobs found")
  return current_job_openings


In [24]:
system_prompt = """You are a Skill-to-Career Mapping assistant that helps students understand skill demand and find matching job opportunities.

You have access to these tools:
- skill_demand_tool: Search for industry demand, salary insights, and career trends
- search_jobs: Find actual job listings requiring specific skills

Help the student by researching the skill they ask about and finding relevant opportunities.

Present results in a clean, readable format with clear sections and proper spacing. Include all job details with apply links. Don't use markdown format."""

In [25]:
search_jobs.invoke({"skill":"Gen AI","location":"INDIA"})

Job search has been started
Searching job for Gen AI in INDIA
10 Jobs found


[{'company_name': 'D E Shaw & Co.',
  'job_title': 'Gen AI Specialist',
  'job_description': 'D E Shaw is hiring for the role of Gen AI Specialist!\n\nResponsibilities of the Candidate:\n• In this role, you will test and integrate LLMs and other foundation models for domain-specific applications (portfolio risk analytics, workflow orchestration and automation, scenario modeling). You will implement RAG (retrieval-augmented generation), tool-use pipelines, MCP components and orchestration layers for end-to-end AI workflows. You will build APIs, data pipelines, and microservices to support on-demand, API-driven access to curated datasets for diverse analytics and AI workloads. You will develop scalable solutions for multi-domain data ingestion, knowledge graphs, and prompt management. Additionally, you will collaborate with quants, risk analysts, and engineers to embed GenAI into systems such as scenario analysis, stress testing, and internal productivity tools. You will also prototype d

In [26]:
agent= create_agent(
  model = model,
  tools = [skill_demand_tool],
  system_prompt = system_prompt,
)

In [27]:
user_query = "What's the demand for generative ai in the industry and show me related job openings in India"

response = agent.invoke({
    "messages": [{"role": "user", "content": user_query}]
})
pprint(response["messages"][-1].content)

[{'extras': {'signature': 'CuoLAb4+9vskP+KOj/CALVpDhRi2IyJMsyXWEJypmtMeStCso5gwDTyG4OKmwWH5/KKlIKR89V+LuJRSVwALD3zGdQATaKgdfx0HVWdbzGPYpaIpdwMZRvlqgpeRp7eAQxH4JQk3eo5/MqacdIIg4Oa7w4RYorpr7rL+57YQ2CtrpOfNltzCybWzHn/7aec+2ByvTiRyZx/8YKMsxU6+hk1B5G1HObuK0DdjcrLso1LS/bt0WZOvNind0L1KOz5P+VFgYitHsYfotH43uPcLvYwEmhyze0scFkMLDWh3aAWAFNAGfl5GhnlduS6N/9+cNm/PDF2l1v83uhTWTDr+mBLgwDO17Tw5bc4gYH8abdstVV58Q0hpCtV/c1+dx0GeZxXghkGT0//8XN6j982kGXiJKzhCeqZ8iHLcMW9F/kOkAH4n4T26bF6gb2Fh2JSlfG7nMqwPV+7pymbg5NNhGTlnwknc47/6h4f8CDP2eE3v/Ym4jx0n6zij6xKjcxsvgiS9mM8sGQZjn+3vwxf5tjdT4hYj8JZfqO7aGn00pZ3UX9L2OiQRyv7Zr3vYUXRSUh5WgDk/uhRNixCqzcuWHRr7B0RC0TCDc5NNn0mRU9narxcY0xbuhyNiAFim2zGFpF+9+7PCYz9W/n7s5SbY6foEafYbfddhBrzv4bRONHXFm+uCRaLRcl0nqu3CPDgSzQTfBmAe/GFC3idlCM013NAWZJIMSV1x56mUT4kmXR4TKSYcVKooHNlBlsrj2I4mXHKl7Tfj1y9zmVZ88ufYoBphP5CDgcibWJd8migUnqTQ/tVkoBLIDLIzAC1/BTtXh2S9SRFVDnR+JmN1cVdFfEo9Q0WOVKB43Un/Sk3VlJGt9vD4mELr7Z4y4zt9YL1hjiqYGMGYfZbqUdp4VUC34+4vHroOxZu8/tYtdALYVYxuFpUjPvhdE4d7xJYXLncAX1PBX885cDcPO